In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np

# dataset loading
train_df = pd.read_csv('/content/drive/MyDrive/DL Lab/lab 1/titanic/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/DL Lab/lab 1/titanic/test.csv')

# dataset preprocessing
def preprocess(df):
    df = df.copy()

    #fill missing value
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # drop unnecessary colum
    df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

    df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

# split feature
X_train = train_df.drop('Survived', axis=1).values.astype(float)
y_train = train_df['Survived'].values

X_test = test_df.values.astype(float)

#feature scaling
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)

X_train = (X_train - X_min) / (X_max - X_min + 1e-8)
X_test = (X_test - X_min) / (X_max - X_min + 1e-8)

#activation function
def step_function(x):
    return 1 if x >= 0 else 0

#perceptron model
class Perceptron:
    def __init__(self, lr=0.01, epochs=50):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0

        for epoch in range(self.epochs):
            for i in range(X.shape[0]):
                linear_output = np.dot(X[i], self.weights) + self.bias
                y_pred = step_function(linear_output)

                # Update rule
                error = y[i] - y_pred
                self.weights += self.lr * error * X[i]
                self.bias += self.lr * error

    def predict(self, X):
        return np.array([
            step_function(np.dot(x, self.weights) + self.bias)
            for x in X
        ])

#train
model = Perceptron(lr=0.01, epochs=100)
model.fit(X_train, y_train)

#predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

#accuracy
accuracy = np.mean(train_pred == y_train)

print("Training Accuracy:", accuracy)
print("\nFirst 30 Test Predictions:")
print(test_pred[:30])

Training Accuracy: 0.7239057239057239

First 30 Test Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0]
